# Local DPO fine-tuning and Ollama deployment

This notebook trains a **LoRA adapter locally** with Hugging Face TRL's `DPOTrainer`, merges it into the downloaded base model, converts the merged model to GGUF, and imports it into Ollama. Ollama runs inference locally; it does not perform DPO training.

The provided `pharma_preference_data.csv` has only a few examples. Treat this as a workflow demo, not a useful medical model. Collect and review substantially more high-quality preference pairs before relying on training results.

On Windows with an AMD GPU, use the **"Python (llms-dml, DirectML)"** kernel. If a cell fails, see **Troubleshooting: issues and how they were fixed** at the end of the notebook.

## Workflow at a glance

1. Load preference examples containing a `prompt`, a preferred `chosen` answer, and a less-preferred `rejected` answer.
2. Load the base language model and train small LoRA weight updates with DPO. The base model is 4-bit only when CUDA and `USE_4BIT` are enabled. On an AMD/Intel GPU under Windows (DirectML, `llms-dml` kernel) the frozen base is FP16 and the model defaults to Qwen2.5-0.5B.
3. Save the adapter, reload the original base model at FP16 on CUDA or FP32 on CPU (DirectML also merges on the CPU), and merge the adapter into it.
4. Convert the merged model to an F16 GGUF file and import that file into Ollama.
5. Ask Ollama a smoke-test question. This only checks that inference works; it does not evaluate medical accuracy.


## The bit-width question

A bit width describes how many bits are used to represent a model weight. Lower-bit representations usually reduce weight memory, but introduce approximation error. Actual memory also includes scales/metadata, activations, gradients, optimizer state, temporary buffers, and runtime overhead, so the table is a weight-only estimate.

| Format | Approx. bytes per weight | Approx. storage for 1.5B weights | Used in this notebook |
| --- | ---: | ---: | --- |
| FP32 (32-bit) | 4 | 6 GB | CPU model load and CPU merge (also used by DirectML) |
| FP16 or BF16 (16-bit) | 2 | 3 GB | CUDA compute and CUDA merge; frozen base on DirectML; GGUF export is F16 |
| 8-bit | 1 | 1.5 GB | Not selected by the current code |
| 4-bit | 0.5 | 0.75 GB before overhead | Optional CUDA QLoRA base-model loading only |

**Important:** 4-bit loading and 16-bit compute can happen together. In this notebook `load_in_4bit=True` stores the frozen base weights in 4-bit NF4, while `bnb_4bit_compute_dtype` selects FP16 or BF16 for calculations. On CPU, the base is loaded in FP32 and `USE_4BIT` defaults to false. On DirectML, the frozen base is loaded in FP16 because the integrated GPU's ~6.5 GB memory can't hold FP32 weights plus training memory; the trainable LoRA weights stay FP32. During merge, the code loads a fresh, unquantized base in FP16 on CUDA or FP32 on CPU. The later `--outtype f16` makes the GGUF export 16-bit; neither that conversion nor Ollama's import changes this notebook into a 4-bit export.

In [ ]:
# Requires trl<=1.14.x on Windows: trl 1.15's DPOTrainer needs Triton, which only ships on Linux.
#   pip install "trl==1.14.2" transformers peft datasets accelerate gguf sentencepiece
# AMD Radeon GPU on Windows (DirectML): use a separate env, because torch-directml pins torch==2.4.1:
#   conda create -n llms-dml python=3.11
#   pip install torch-directml "torch==2.4.1" "transformers==4.56.1" trl peft accelerate "datasets==3.6.0" pandas gguf sentencepiece ipykernel
from pathlib import Path
import gc
import os
import subprocess
import sys
import torch
from datasets import Dataset, load_dataset
from huggingface_hub.constants import HF_HUB_CACHE
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, TaskType
from trl import DPOConfig, DPOTrainer

# Device selection: CUDA (NVIDIA) > DirectML (AMD/Intel GPU on Windows, e.g. ASUS GA402R) > CPU.
USE_CUDA = torch.cuda.is_available()
try:
    import torch_directml
    USE_DML = not USE_CUDA and torch_directml.is_available()
except ImportError:
    USE_DML = False

if USE_CUDA:
    DEVICE = "cuda"
elif USE_DML:
    DEVICE = torch_directml.device()
else:
    DEVICE = "cpu"
USE_4BIT = USE_CUDA  # bitsandbytes 4-bit QLoRA needs an NVIDIA GPU
if not USE_CUDA:
    torch.set_num_threads(max(1, os.cpu_count() // 2))  # one thread per physical core

# On a DirectML iGPU, the 1.5B model trips the Windows GPU watchdog (TDR: the driver resets when one
# GPU operation runs longer than ~2 s), so DirectML defaults to the 0.5B model.
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct" if USE_DML else "Qwen/Qwen2.5-1.5B-Instruct"
OLLAMA_MODEL_NAME = "pharma-qwen-dpo"
OUTPUT_DIR = Path("runs") / f"{BASE_MODEL.split('/')[-1].lower()}-dpo"
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_MODEL_DIR = OUTPUT_DIR / "ollama-model"
GGUF_PATH = OUTPUT_DIR / f"{OLLAMA_MODEL_NAME}-f16.gguf"
LLAMA_CPP_DIR = Path.home() / "llama.cpp"  # Folder containing convert_hf_to_gguf.py
DATA_SOURCE = "local"  # Change to "huggingface" to use a Hub preference dataset.
HF_DATASET_ID = "trl-lib/ultrafeedback_binarized"
HF_DATASET_SPLIT = "train"
MAX_TRAIN_SAMPLES = None  # Optional cap for Hub dataset

model_cache_dir = Path(HF_HUB_CACHE) / f"models--{BASE_MODEL.replace('/', '--')}"
print(f"Base model: {BASE_MODEL}")
print(f"Base model cache folder: {model_cache_dir}")
print(f"Training output folder: {OUTPUT_DIR.resolve()}")
device_name = torch_directml.device_name(0).rstrip("\x00") if USE_DML else DEVICE
print(f"Device: {DEVICE} ({device_name}); torch threads: {torch.get_num_threads()}; 4-bit QLoRA: {USE_4BIT}")

## Load and validate preference pairs

DPO learns from comparisons, not from a single answer label: each row says that `chosen` is preferred to `rejected` for the same `prompt`. The local CSV must have those three column names.

This notebook is intended as a local workflow demo; validate the CSV before training and keep the dataset small, clean, and medically cautious.


In [2]:
import pandas as pd

csv_path = Path("pharma_preference_data.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Preference CSV not found: {csv_path}")

df = pd.read_csv(csv_path)
required = {"prompt", "chosen", "rejected"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = df[["prompt", "chosen", "rejected"]].dropna().reset_index(drop=True)
print(f"Loaded {len(df)} preference pairs")
df.head()


Loaded 5 preference pairs


,prompt,chosen,rejected
0,Explain the mechanism of action of Metformin i...,"Metformin activates AMPK, which helps cells us...",Metformin only lowers blood sugar by stopping ...
1,Compare the lipid-lowering effects of Atorvast...,"When used together, Atorvastatin and Ezetimibe...","Both drugs work the same way, so using them to..."
2,Summarize the recent developments in mRNA vacc...,mRNA vaccines can be redesigned rapidly to tar...,mRNA vaccines are identical for all viruses an...
3,Describe how artificial intelligence is used i...,AI accelerates drug discovery by predicting pr...,AI in pharma is mostly used for marketing and ...
4,What are some non-glycemic benefits of Metform...,"Beyond blood sugar control, Metformin has show...",Metformin is strictly an anti-diabetic drug an...


In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# DPOTrainer applies the model's chat template itself when the data is in
# conversational form: a user prompt plus two candidate assistant replies.
def to_conversation(row):
    return {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "chosen": [{"role": "assistant", "content": row["chosen"]}],
        "rejected": [{"role": "assistant", "content": row["rejected"]}],
    }

if DATA_SOURCE == "local":
    train_dataset = Dataset.from_pandas(df).map(to_conversation)
else:
    train_dataset = load_dataset(HF_DATASET_ID, split=HF_DATASET_SPLIT)
    if MAX_TRAIN_SAMPLES:
        train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))

print(f"Training rows: {len(train_dataset)}")
train_dataset[0]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Training rows: 5


{'prompt': [{'content': 'Explain the mechanism of action of Metformin in simple scientific terms.',
   'role': 'user'}],
 'chosen': [{'content': 'Metformin activates AMPK, which helps cells use glucose more efficiently, reduces glucose production by the liver, and improves insulin sensitivity without increasing insulin levels.',
   'role': 'assistant'}],
 'rejected': [{'content': 'Metformin only lowers blood sugar by stopping sugar absorption from food and does not affect other organs.',
   'role': 'assistant'}]}

## Train and save the adapter

The next cells configure the LoRA adapter and DPO trainer, then run the optimization loop. The goal is to update the adapter weights while keeping the base model frozen.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

dpo_config = DPOConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,
    num_train_epochs=1,
    max_length=512,              # keeps CPU memory and time manageable
    use_cpu=not USE_CUDA,        # Accelerate has no DirectML backend; DirectMLDPOTrainer places the model itself
    bf16=False,
    fp16=USE_CUDA,               # mixed precision only on CUDA; CPU/DirectML train in FP32
    gradient_checkpointing=True, # trades speed for lower RAM use
    save_strategy="no",          # the adapter is saved explicitly below
    logging_steps=1,
    report_to="none",
    dataloader_pin_memory=USE_CUDA,
    remove_unused_columns=False,
)


class DirectMLDPOTrainer(DPOTrainer):
    """DPOTrainer that keeps the model on the DirectML device and moves each batch there.

    Hugging Face Trainer/Accelerate only know CPU and CUDA, so with use_cpu=True they would
    pull the model back to the CPU. The model is flagged as already placed, and Accelerate's
    automatic device placement is switched off before training.
    """

    def __init__(self, *args, **kwargs):
        model = kwargs["model"]
        model.is_parallelizable = True  # Trainer skips moving a model it considers already placed
        model.model_parallel = True
        super().__init__(*args, **kwargs)
        self.accelerator.device_placement = False

    def _prepare_inputs(self, inputs):
        return {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}


if USE_DML:
    import trl.trainer.dpo_trainer as dpo_trainer_module

    def dml_selective_log_softmax(logits, index):
        # TRL uses torch.gather here; its backward is a scatter that DirectML rejects.
        # Plain advanced indexing gives the same values and gradients.
        logps = logits.float().log_softmax(-1).reshape(-1, logits.shape[-1])  # FP32 for stability
        flat_index = index.reshape(-1)
        rows = torch.arange(flat_index.numel(), device=flat_index.device)
        return logps[rows, flat_index].reshape(index.shape)

    dpo_trainer_module.selective_log_softmax = dml_selective_log_softmax

TrainerClass = DirectMLDPOTrainer if USE_DML else DPOTrainer
print(f"LoRA and DPO configs prepared; trainer: {TrainerClass.__name__}")

In [ ]:
if USE_4BIT:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        ),
        device_map={"": 0},
    )
else:
    # DirectML on an iGPU gets ~6.5 GB of shared memory: FP32 weights (6 GB) don't fit, so the
    # frozen base is FP16 there. PEFT keeps the trainable LoRA weights in FP32.
    train_dtype = torch.float16 if USE_DML else torch.float32
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=train_dtype).to(DEVICE)
model.config.use_cache = False

# With peft_config, DPOTrainer wraps the model in LoRA and uses the frozen base
# (adapter disabled) as the reference model, so no second model copy is loaded.
trainer = TrainerClass(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

if USE_DML:
    trainer.model.to("cpu")  # safetensors can't read DirectML tensors directly
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"Adapter saved to {ADAPTER_DIR.resolve()}")

In [ ]:
# Free the training model before loading a fresh, unquantized base for merging.
del trainer, model
gc.collect()
if USE_CUDA:
    torch.cuda.empty_cache()

# Merging is a one-off weight addition, so DirectML runs merge on the CPU.
merge_device = "cuda" if USE_CUDA else "cpu"
merge_dtype = torch.float16 if USE_CUDA else torch.float32
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=merge_dtype).to(merge_device)
merged_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).merge_and_unload()

merged_model_dir = MERGED_MODEL_DIR / "merged"
merged_model.save_pretrained(str(merged_model_dir), safe_serialization=True)
tokenizer.save_pretrained(str(merged_model_dir))
print(f"Merged model saved to {merged_model_dir.resolve()} ({merge_dtype})")

del merged_model, base_model
_ = gc.collect()

## Convert to GGUF and import into Ollama

`convert_hf_to_gguf.py` comes from a local llama.cpp checkout (set `LLAMA_CPP_DIR` above) and needs the `gguf` Python package. The export is F16. Ollama reads the chat template from the GGUF metadata.

In [ ]:
convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
if not convert_script.exists():
    raise FileNotFoundError(f"llama.cpp converter not found: {convert_script}")

# The converter imports sentencepiece even for BPE tokenizers like Qwen's, so it must be installed.
result = subprocess.run(
    [sys.executable, str(convert_script), str(merged_model_dir),
     "--outfile", str(GGUF_PATH), "--outtype", "f16"],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
)
if result.returncode != 0:
    # Without this, the notebook only shows "exit status 1"; the reason is in the converter's stderr.
    raise RuntimeError(f"GGUF conversion failed:\n{result.stderr[-3000:]}")
print(f"GGUF written to {GGUF_PATH.resolve()}")

In [ ]:
modelfile_path = OUTPUT_DIR / "Modelfile"
modelfile_path.write_text(
    f"FROM {GGUF_PATH.resolve().as_posix()}\n"
    "PARAMETER temperature 0.2\n"
    'SYSTEM "You are a careful pharmacology assistant. You are not a substitute for a clinician."\n',
    encoding="utf-8",
)
subprocess.run(["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(modelfile_path)], check=True)
print(f"Ollama model created: {OLLAMA_MODEL_NAME}")

In [ ]:
import json
import urllib.request

OLLAMA_MODEL_NAME = "pharma-qwen-dpo"

# Smoke test only: checks that inference runs, not that the answers are medically correct.
# Calls the local Ollama server's HTTP API instead of the `ollama run` CLI, which is built for
# interactive terminals and can hang inside a Jupyter kernel.
request = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=json.dumps({
        "model": OLLAMA_MODEL_NAME,
        "prompt": "Explain the mechanism of action of Metformin in two sentences.",
        "stream": False,
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=300) as response:
    print(json.loads(response.read())["response"])

## Important note

This notebook is a workflow demo and not a medically reliable model. It is meant to show the local DPO + GGUF + Ollama pipeline, not to provide medical advice.


## Troubleshooting: issues and how they were fixed

Running this notebook on a Windows laptop with an AMD Radeon GPU (ASUS GA402R, 16 GB RAM) hit 11 problems. Each was fixed in the cells above or in the environment. They are listed in the order they appeared; each fix exposed the next one.

**Which kernel to use**

| Kernel | Environment | PyTorch | Trains on | Model |
| --- | --- | --- | --- | --- |
| Python (llms-dml, DirectML) | `anaconda3\envs\llms-dml`, Python 3.11 | 2.4.1 + torch-directml 0.2.5 | Radeon GPU | Qwen2.5-0.5B |
| llms | `anaconda3\envs\llms`, Python 3.11 | 2.14.1 (CPU) | CPU (slow) | Qwen2.5-1.5B |

Install packages for a notebook with `%pip install ...` in a cell, or with that environment's own `python.exe -m pip`. A plain `pip` in the terminal installs into the global Python, which this notebook doesn't use.

**Summary**

| # | Symptom | Cause | Fix |
| --- | --- | --- | --- |
| 1 | `ModuleNotFoundError: No module named 'trl'` | trl wasn't installed in the kernel's environment | Installed trl into `llms` |
| 2 | `ImportError: ... requires accelerate>=0.26.0` | Kernel kept a stale "not installed" answer from before the install | Restart the kernel |
| 3 | Training ran on the CPU | CUDA is NVIDIA-only; this laptop has an AMD GPU | Use DirectML (`torch-directml`) |
| 4 | torch-directml needs `torch==2.4.1` | Installing it would downgrade PyTorch in `llms` | Separate `llms-dml` environment |
| 5 | `unbox expects Dml at::Tensor as inputs` | Trainer/Accelerate moved the model back to the CPU | `DirectMLDPOTrainer` subclass |
| 6 | `DirectML scatter doesn't allow partially modified dimensions` | Backward pass of `torch.gather` in trl | Replaced with plain indexing |
| 7 | `The GPU will not respond to more commands` while loading | FP32 1.5B model (6 GB) exceeds the ~6.5 GB GPU budget | FP16 base on DirectML |
| 8 | Same error during training of the 1.5B model | Likely the Windows GPU watchdog (TDR) | Qwen2.5-0.5B on DirectML |
| 9 | `Cannot access storage of OpaqueTensorImpl` on save | safetensors can't read DirectML tensors | Move the model to the CPU before saving |
| 10 | GGUF conversion: `exit status 1` | `sentencepiece` missing for llama.cpp's converter | Installed it; cell now shows the real error |
| 11 | Ollama smoke test runs forever | `ollama run` CLI hangs without a terminal | Call Ollama's HTTP API instead |

### 1. trl is not installed

- **Error:** `ModuleNotFoundError: No module named 'trl'` on `from trl import DPOConfig, DPOTrainer`.
- **Cause:** the "llms" kernel is the conda environment at `C:\Users\sridh\anaconda3\envs\llms`, and trl was never installed there.
- **Fix:** installed trl 0.26.2 into `llms`, pinning `transformers==4.56.1` and `datasets==3.6.0` so pip couldn't upgrade them.

### 2. accelerate reported as missing

- **Error:** `ImportError: Using the Trainer with PyTorch requires accelerate>=0.26.0`, raised by `DPOConfig(...)`.
- **Cause:** accelerate 1.15.0 was installed. transformers checks for accelerate once, when it is first imported, and keeps that answer for the life of the kernel. A kernel started before an install keeps the stale answer.
- **Fix:** restart the kernel and run from the top. No package change was needed.

### 3. No CUDA on an AMD laptop

- **Symptom:** `torch.cuda.is_available()` is `False`, so everything trained on the CPU.
- **Cause:** CUDA only works on NVIDIA GPUs. AMD's own PyTorch backend for Windows (ROCm) supports only RX 7000-series and newer GPUs.
- **Fix:** use DirectML, Microsoft's GPU backend for Windows, through the `torch-directml` package. The setup cell now picks CUDA first, then DirectML, then the CPU.

### 4. torch-directml needs an old PyTorch

- **Problem:** the latest torch-directml (0.2.5.dev240914) requires exactly `torch==2.4.1`, but `llms` has torch 2.14.1.
- **Fix:** a separate conda environment, `llms-dml`, registered as the kernel "Python (llms-dml, DirectML)". The commands to build it are in the setup cell's header comments.

### 5. Trainer moves the model back to the CPU

- **Error:** `RuntimeError: ... unbox expects Dml at::Tensor as inputs`, in the embedding layer.
- **Cause:** Hugging Face Trainer and Accelerate only understand CPU and CUDA devices. With `use_cpu=True`, Accelerate's `prepare()` moved the model back to the CPU when training started, while the batch was on the GPU.
- **Fix:** the `DirectMLDPOTrainer` class in the config cell. It marks the model as already placed (`is_parallelizable` and `model_parallel`), turns off Accelerate's automatic device placement, and moves each batch to the GPU in `_prepare_inputs`.

### 6. Backward pass fails in gather

- **Error:** `RuntimeError: DirectML scatter doesn't allow partially modified dimensions`.
- **Cause:** trl's `selective_log_softmax` uses `torch.gather` to pick each token's log-probability. Its gradient is a scatter that DirectML doesn't support in this shape.
- **Fix:** on DirectML, the config cell replaces that function with `dml_selective_log_softmax`, which uses plain indexing (`logps[rows, index]`). A test showed identical values and gradients.

### 7. FP32 model doesn't fit in GPU memory

- **Error:** `RuntimeError: The GPU will not respond to more commands`, while copying the model to the GPU.
- **Cause:** the integrated GPU borrows system RAM, and DirectML could allocate about 6.5 GB. Qwen2.5-1.5B in FP32 needs 6 GB for the weights alone.
- **Fix:** on DirectML the frozen base model loads in FP16 (`train_dtype` in the training cell). PEFT keeps the trainable LoRA weights in FP32, and log-probabilities are computed in FP32 for stability.

### 8. GPU driver resets on the 1.5B model

- **Error:** the same `The GPU will not respond to more commands`, now during training, even with shorter sequences.
- **Likely cause:** the Windows GPU watchdog (TDR) resets the driver when one GPU operation runs longer than about 2 seconds. The same code trains cleanly on smaller models, which points at operation size rather than a code bug. This is inferred, not confirmed.
- **Fix:** on DirectML, `BASE_MODEL` defaults to `Qwen/Qwen2.5-0.5B-Instruct`. Use the "llms" kernel to train the 1.5B model on the CPU.

### 9. Saving the adapter fails

- **Error:** `NotImplementedError: Cannot access storage of OpaqueTensorImpl`, in `trainer.save_model`.
- **Cause:** safetensors reads tensor memory directly, which DirectML tensors don't allow.
- **Fix:** the training cell moves the model to the CPU before saving, and the merge cell runs on the CPU.

### 10. GGUF conversion fails with exit status 1

- **Error:** `CalledProcessError: ... convert_hf_to_gguf.py ... returned non-zero exit status 1`, with no reason shown.
- **Cause:** running the converter directly showed `ModuleNotFoundError: No module named 'sentencepiece'`. llama.cpp's Qwen converter imports `sentencepiece` before falling back to Qwen's real tokenizer, so the import fails first.
- **Fix:** installed `sentencepiece` into both environments. The conversion cell now captures the converter's output and shows its error message if it fails.

### 11. Ollama smoke test runs forever

- **Symptom:** the smoke-test cell kept loading for over 6 minutes. In the Ollama app, the same question answered immediately.
- **Cause:** the `ollama run` command is built for an interactive terminal. Inside the Jupyter kernel it waited for input that never came; `ollama ps` showed no model loaded. Passing `stdin=subprocess.DEVNULL` fixed this in a normal Python process but not in the VS Code kernel.
- **Fix:** the smoke-test cell now calls the Ollama server's HTTP API (`http://localhost:11434/api/generate`) with `urllib`. It answers in about 5 seconds, with clean text and no terminal control codes. Stop any leftover stuck process with `Stop-Process -Id <pid>` (find it with `Get-Process ollama`).

### Results

- **DirectML, Qwen2.5-0.5B:** training took about 9 seconds; loss fell from 0.693 to 0.613, and all 96 LoRA layers were updated. Saving and merging worked.
- **CPU, Qwen2.5-1.5B:** training, merge, GGUF conversion (3.1 GB F16 file), `ollama create` and the smoke test all worked.

### Known limits

- The DirectML fixes rely on internals of trl (`selective_log_softmax`) and of Trainer (`_prepare_inputs` and its model-placement checks). After upgrading trl or transformers in `llms-dml`, rerun the notebook to confirm it still works.
- With only 5 preference pairs, training shows that the pipeline works, not that the model improved.